# 3. Data Analysis - Indicators

In [1]:
# import packages
import os
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
# set data raw path (where files were downloaded)
data_path = '/Users/carla_q/Library/CloudStorage/GoogleDrive-kchen@datapopalliance.org/My Drive/Morocco care mapping/use case 2/data_processed'
# set data pre-processed path (where the pre-processed files will be saved)
output_path = data_path + '/indicator'

## 3.1 Indicator calculation

In [3]:
# population related indicators
gdf_path = data_path + '/zonalforstat/zonal_for_stat.shp'
gdf = gpd.read_file(gdf_path)

gdf['pop_density_0_4'] = gdf['pop_0_4'] / gdf['AREA_SQKM']
gdf['child_dep_ratio'] = gdf['pop_0_4'] / gdf['pop_f15_59'] * 1000

In [4]:
# facility related indicators
supply_path = data_path + '/facility/gdf_facility.shp'
supply = gpd.read_file(supply_path)

supply = supply.to_crs(gdf.crs)
supply.loc[supply['Data Sourc']!= 'Public', 'Data Sourc'] = 'Private'

# calculate childcare facilities by data source and by category
joined = gpd.sjoin(supply, gdf, how="left", predicate="within")

grouped = (
    joined.groupby(['ADM3_PCODE', 'Data Sourc', 'Category'])
    .size()
    .reset_index(name='count')
)

source_counts = grouped.pivot_table(
    index='ADM3_PCODE',
    columns='Data Sourc',
    values='count',
    aggfunc='sum',
    fill_value=0
)

category_counts = grouped.pivot_table(
    index='ADM3_PCODE',
    columns='Category',
    values='count',
    aggfunc='sum',
    fill_value=0
)

gdf = gdf.join(source_counts, how='left', on='ADM3_PCODE')
gdf = gdf.join(category_counts, how='left', on='ADM3_PCODE')

gdf = gdf.fillna(0)
gdf['childcare_total'] = gdf['Private'] + gdf['Public'] 

# calculate share of municipal childcare, and coverage ratio
gdf['share_public_childcare'] = np.where(gdf['childcare_total'] > 0, gdf['Public'] / gdf['childcare_total'], 0)
gdf['childcare_coverage_ratio'] = gdf['childcare_total'] / gdf['pop_0_4'] * 1000

In [5]:
# Socioeconomic indicators
soc_path = data_path.replace('data_processed', 'data_raw') + '/employment_poverty/2024_census_emp_pov.csv'
df = pd.read_csv(soc_path)
df.columns = ['ADM3_AR', 'ADM3_FR', 'm_unemp_rate', 'f_unemp_rate', 'total_unemp_rate', 'mpi']
df['unemp_gap'] = df['f_unemp_rate'] - df['m_unemp_rate']
gdf = gdf.merge(df.drop('ADM3_AR',axis=1), how='left', on='ADM3_FR')

## 3.2 Export Indicators

In [6]:
gpkg_path = data_path.replace('data_processed', 'indicator') + '/indicator.gpkg'
csv_path = data_path.replace('data_processed', 'indicator') + '/indicator.csv'

gdf.to_file(gpkg_path, driver="GPKG")
gdf.drop(columns="geometry").to_csv(csv_path, index=False)